In [ ]:
# Load the AAC research 7000+ vocabulary
from google.colab import drive
import json
import re
drive.mount('/content/drive', force_remount=True)

csv_path = '/content/drive/MyDrive/corpus_vocabulary_slim.csv'


import pandas as pd
df = pd.read_csv(csv_path)

name_to_idx = {n.lower(): i for i, n in enumerate(df['name_en'])}

print(f'categories: {df["category"].nunique()}   total words: {len(df)}')

Mounted at /content/drive
categories: 23   total words: 6658


In [ ]:
# Split into singleWords and multiWords
from numpy._core import multiarray
singleWords = []
multiWords = []
for row in df.itertuples():
  if len(row.name_en.split()) == 1:
    singleWords.append(row[1:])
  else:
    multiWords.append(row[1:])
print(multiWords[3])
multiWords = pd.DataFrame(multiWords, columns = df.columns)
singleWords = pd.DataFrame(singleWords, columns = df.columns)
# Strips out all hidden backspace characters
singleWords['name_en'] = singleWords['name_en'].str.replace('\x08', '', regex=False).str.strip()
multiWords['name_en'] = multiWords['name_en'].str.replace('\x08', '', regex=False).str.strip()

('animal', 'Animal Feed', 'A transparent jar with a yellow lid, filled with numerous small, brown, cube-shaped objects, resembling animal feed or pellets. A couple of pellets appear to have spilled outside the jar, resting next to it. The image conveys the concept of a container with pet or livestock feed.')


In [ ]:
# Building a dedupe set
seen = set()
for i in singleWords['name_en']:
  seen.add(i.lower())
print("cheating" in seen)
print(seen)

True
{'everyone', 'several', 'green', 'doctor/physician', '72', 'lens', 'the-more', 'pineapple', 'games', 'dream', 'clergyman', 'kindergarten', 'collision', '70', 'adidas', 'zipper', 'equal', 'father-in-law', 'crutches', 'robot', 'guarantor', 'measurement', 'where?', 'chicks', 'u-turn', 'personality', 'diabetes', 'rain', 'democracy', 'ointment', 'thoughtful', 'dolphin', 'cheeseburger', 'snacks', 'anchor', 'moonlight', 'manager', 'waiting', 'supermarket', 'croquet', 'potato', 'grilling', 'printer', 'nesquik', 'hoodie', 'wolf', 'seashell', 'candy', 'omurice', 'yesterday', 'golf', 'koala', 'wow', 'danger', 'centipede', 'mailbox', 'sick', 'bulgogi', 'yes', 'scoot', 'pistachio', 'organization', 'mosquito', 'president', 'stadium', 'refill', 'april', 'tomorrow', 'laundry', '94', 'id', 'write', 'go', 'bee', 'selling', 'windows', 'swan', 'joystick', 'travel', 'factory', 'bible', 'arguing', 'button', 'espresso', 'design', 'mangoes', '99', 'timer', 'milk', 'when?', 'daffodils', 'team', 'squid', '

In [ ]:
# Using an LLM to change multi words to single words
results = []
from openai import OpenAI
client = OpenAI(
    api_key="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",
    base_url='https://openrouter.ai/api/v1',
)
def gen_words(batch):
  text = ""
  lines = [f"{i}. category: {row['category']}, name: {row['name_en']}" for i, row in
  enumerate(batch)]
  user_message = "Process these entries:\n" + "\n".join(lines)
  system_prompt = ("""
  You are a vocabulary processor for an AAC system where non-verbal users
  tap single-word cards to communicate (e.g., 'school eat happy').

  Given a list of multi-word vocabulary entries, extract the single most
  useful English word for each.

  RULES:
  - Return one lowercase English word, or SKIP
  - Valid word types: nouns, verbs, adjectives, emotion words
  - SKIP if: full sentence, question, instruction, or no single word fits
  - Never return prepositions, articles, or conjunctions
  - For foreign terms with English translations, use the English word
  - Prefer specific over generic ('cheese' not 'food')

  Return ONLY a JSON array — no other text:
  [{"index": 0, "word": "cheese"},
   {"index": 1, "word": "weight"},
   {"index": 2, "word": "SKIP"}]
  """
  )
  response = client.chat.completions.create(
    model='google/gemini-2.5-flash-lite',
    max_tokens = 4096,
    messages=[
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_message}
          ]
      )
  text = response.choices[0].message.content
  text = re.sub(r"```json|```", "", text).strip()
  try:
    return json.loads(text)
  except json.JSONDecodeError as e:
    print(f"JSON parse failed: {e}")
    print(f"Raw response: {text[:500]}")
    return []


In [ ]:
# Make batches from multiWord
new_rows = []
BATCH_SIZE = 100
for i in range(0, len(multiWords), BATCH_SIZE):
  batch_num = i // BATCH_SIZE + 1
  batch = multiWords.iloc[i:i+BATCH_SIZE].to_dict('records')
  lookup = {i: row for i, row in enumerate(batch)}
  try:
    results = gen_words(batch)
  except Exception as e:
    print(f"Batch {i // BATCH_SIZE + 1} failed: {e}")
    continue
  skipped = 0
  duped = 0
  accepted = 0
  for result in results:
    word = result.get("word")
    index = result.get("index")
    if word == "SKIP":
      skipped += 1
      continue
    if word is None or index is None:
      continue
    if word.lower() in seen:
      duped += 1
      continue
    accepted += 1

    original_row = lookup.get(index)
    if original_row is None:
      continue
    seen.add(word.lower())

    new_rows.append({
        "category": original_row['category'],
        "name_en": word,
        "description_brief": original_row['description_brief'],
    })
    with open('new_rows_checkpoint.json', 'w') as f:
      json.dump(new_rows, f)
  print(f"Batch {batch_num} done, {len(new_rows)} words so far")
  print(f"Batch {batch_num} — accepted: {accepted}, skipped: {skipped}, duped: {duped}")


Batch 1 done, 4 words so far
Batch 1 — accepted: 4, skipped: 15, duped: 81
Batch 2 done, 4 words so far
Batch 2 — accepted: 0, skipped: 14, duped: 86
Batch 3 done, 4 words so far
Batch 3 — accepted: 0, skipped: 34, duped: 66
Batch 4 done, 4 words so far
Batch 4 — accepted: 0, skipped: 19, duped: 81
Batch 5 done, 5 words so far
Batch 5 — accepted: 1, skipped: 69, duped: 30
Batch 6 done, 5 words so far
Batch 6 — accepted: 0, skipped: 69, duped: 31
Batch 7 done, 6 words so far
Batch 7 — accepted: 1, skipped: 14, duped: 85
Batch 8 done, 7 words so far
Batch 8 — accepted: 1, skipped: 26, duped: 73
Batch 9 done, 8 words so far
Batch 9 — accepted: 1, skipped: 12, duped: 87
Batch 10 done, 8 words so far
Batch 10 — accepted: 0, skipped: 50, duped: 50
Batch 11 done, 8 words so far
Batch 11 — accepted: 0, skipped: 7, duped: 93
Batch 12 done, 8 words so far
Batch 12 — accepted: 0, skipped: 7, duped: 93
Batch 13 done, 9 words so far
Batch 13 — accepted: 1, skipped: 13, duped: 86
Batch 14 done, 9 wo

In [ ]:
len(new_rows)

56

In [ ]:
new_rows_df = pd.DataFrame(new_rows)
new_rows_df = new_rows_df[~new_rows_df['name_en'].str.contains(' ', na=False)]
final_df = pd.concat([singleWords, new_rows_df], ignore_index = True)
final_df.to_csv('corpus_vocabulary_clean.csv', index = False)
print(f"Saved {len(final_df)} words")

Saved 1998 words
